# Capstone: Mathematical Analysis of GPT

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 12.03 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/04_mathematical_analysis_of_gpt.ipynb)

---

## 🎯 Learning Objective

Count parameters and FLOPs for a GPT-style model and derive its memory requirements, connecting the numbers to published scaling laws.

---

## 📐 Theory

`05.07` used the Chinchilla scaling law to answer "given a compute budget, how should I split it
between model size $N$ and data size $D$" — treating $N$ as a single number fed into a fitted
formula. This capstone opens up that number: where does $N$ actually come from, architecturally,
how many floating-point operations does using it cost, and how much memory does it actually
require to train or serve. These are the numbers that turn an architecture on paper into a
concrete hardware requirement.

### Counting parameters

A GPT-style transformer's parameters are dominated by the per-layer weight matrices. Each layer
has a self-attention block ($W_Q,W_K,W_V,W_O$, each $d_{\text{model}}\times d_{\text{model}}$ —
recall `01.10`) and a feed-forward block (two matrices, $d_{\text{model}}\times d_{ff}$ and
$d_{ff}\times d_{\text{model}}$, typically with $d_{ff}=4d_{\text{model}}$). This gives, per
layer, $4d_{\text{model}}^2 + 2d_{\text{model}}d_{ff} = 4d_{\text{model}}^2 +
8d_{\text{model}}^2 = 12\,d_{\text{model}}^2$ parameters (ignoring biases and LayerNorm, which
are a small correction). Across $n_{\text{layers}}$ layers, plus a token embedding table
($V\times d_{\text{model}}$, often **weight-tied** with the final output projection so it's
counted only once), this gives a closed-form parameter count — verified below against a real
PyTorch model's *actual* `numel()` count, not just the formula's own self-consistency.

### FLOPs: the "$2N$" and "$6N$" rules

For a single input token, a forward pass through one linear layer $W\in\mathbb{R}^{d_{in}\times
d_{out}}$ costs $2\,d_{in}d_{out}$ floating point operations (one multiply and one add per
weight, the standard FLOP-counting convention — recall matrix-vector products from `01.02`).
Summing this over every matmul in every layer gives total forward FLOPs-per-token that equals
**exactly** $2N$ (where $N$ is the matmul parameter count from above) — not an approximation,
verified below by explicit counting. Backpropagation (recall `02.10`) costs roughly twice the
forward pass's FLOPs (one pass to get gradients w.r.t. inputs, another w.r.t. weights), so a
full training step costs approximately $6N$ FLOPs per token — the exact quantity `05.07`'s
Chinchilla formula $C\approx 6ND$ uses, now derived rather than taken as given.

### Memory: training vs. inference

**Training memory** (using Adam, recall `04.04`/`12.03`) needs to store, per parameter: the
parameter itself, its gradient, Adam's first moment $m$, and second moment $v$ — four
same-sized buffers. At fp32 (4 bytes each), that's $16$ bytes per parameter, confirmed to match
published estimates for real models. **Inference memory** only needs the weights themselves
(often at fp16/bf16, 2 bytes each) — an order-of-magnitude smaller footprint, since there's no
gradient or optimizer state to keep. Inference has one more cost training doesn't:
the **KV cache** (recall `01.10`'s $K,V$ matrices) — every generated token requires storing that
token's keys and values for every layer and head, so they don't need recomputing for every
future token; this cache grows linearly with sequence length and batch size, and can become the
dominant memory cost for long-context, high-batch-size serving.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Breaking down the parameter-counting formula's three components (attention, feed-forward,
embedding) across the four real, published GPT-2 model sizes shows how the *embedding* table's
share shrinks as the model gets deeper and wider — it's fixed by vocabulary size, while the
per-layer matmul cost grows with $d_{\text{model}}^2$.

In [ ]:
# Real, published GPT-2 family configurations
gpt2_configs = {
    "GPT-2 small\n(124M)":  dict(n_layers=12, d_model=768,  vocab=50257),
    "GPT-2 medium\n(355M)": dict(n_layers=24, d_model=1024, vocab=50257),
    "GPT-2 large\n(774M)":  dict(n_layers=36, d_model=1280, vocab=50257),
    "GPT-2 XL\n(1.5B)":     dict(n_layers=48, d_model=1600, vocab=50257),
}

names, attn_list, ff_list, embed_list, total_list = [], [], [], [], []
for name, cfg in gpt2_configs.items():
    d_model, n_layers, vocab = cfg["d_model"], cfg["n_layers"], cfg["vocab"]
    d_ff = 4 * d_model
    attn = n_layers * 4 * d_model**2
    ff = n_layers * 2 * d_model * d_ff
    embed = vocab * d_model
    names.append(name)
    attn_list.append(attn / 1e6); ff_list.append(ff / 1e6); embed_list.append(embed / 1e6)
    total_list.append((attn + ff + embed) / 1e6)

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(names))
ax.bar(x, attn_list, label="attention ($4 n_L d_{model}^2$)", color="#4C72B0")
ax.bar(x, ff_list, bottom=attn_list, label="feed-forward ($8 n_L d_{model}^2$)", color="#DD8452")
bottom2 = np.array(attn_list) + np.array(ff_list)
ax.bar(x, embed_list, bottom=bottom2, label="embedding ($V \\cdot d_{model}$, tied)", color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("parameters (millions)")
ax.set_title("Formula-predicted parameter breakdown matches published\nGPT-2 model sizes (124M / 355M / 774M / 1.5B) closely")
ax.legend()
plt.tight_layout()
plt.show()

for name, total, embed in zip(names, total_list, embed_list):
    print(f"{name.split(chr(10))[0]}: formula total={total:.1f}M params, embedding share={100*embed/total:.1f}%")

## 🐍 Implementation from Scratch

We build an actual GPT-2-small-scale transformer in PyTorch and confirm the parameter-counting
formula matches its real `numel()` count almost exactly, verify the "$2N$ forward FLOPs" claim
by explicit per-operation counting (not approximation), and compute realistic training vs.
inference memory requirements including the KV cache.

In [ ]:
# --- Build an ACTUAL GPT-2-small-scale transformer and count its real parameters ---
class GPTBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn_qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.attn_out = nn.Linear(d_model, d_model, bias=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff1 = nn.Linear(d_model, d_ff, bias=True)
        self.ff2 = nn.Linear(d_ff, d_model, bias=True)

class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # standard GPT-2 weight tying

def count_params_unique(model):
    # counts each UNIQUE parameter tensor once, correctly handling the tied embedding/head
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p)); total += p.numel()
    return total

vocab_size, d_model, n_layers, d_ff, max_seq_len = 50257, 768, 12, 3072, 1024
model = GPTModel(vocab_size, d_model, n_layers, d_ff, max_seq_len)
actual_params = count_params_unique(model)

# Closed-form formula: 12 * n_layers * d_model^2 (attn+ff matmuls) + tied embedding + position embedding
N_matmuls = n_layers * (4 * d_model**2 + 2 * d_model * d_ff)
N_formula = N_matmuls + vocab_size * d_model + max_seq_len * d_model

print(f"GPT-2-small-scale model (d_model={d_model}, n_layers={n_layers}, vocab={vocab_size}):")
print(f"  Actual PyTorch param count (unique tensors, tied weights counted once): {actual_params:,}")
print(f"  Formula (12*n_layers*d_model^2 + tied embed + pos embed):               {N_formula:,}")
print(f"  Relative difference: {abs(N_formula - actual_params)/actual_params:.4f} "
      f"(remaining gap: biases + LayerNorm params, ignored by the simplified formula)")
print(f"  This matches OpenAI's published GPT-2-small size of ~124M parameters.")

# --- FLOPs: verify the "2N" forward-pass identity by EXPLICIT per-operation counting ---
def linear_flops(d_in, d_out):
    return 2 * d_in * d_out  # one multiply + one add per weight (standard FLOP convention)

per_layer_flops = (linear_flops(d_model, d_model) * 3    # Q, K, V projections
                    + linear_flops(d_model, d_model)      # output projection
                    + linear_flops(d_model, d_ff)          # ff1
                    + linear_flops(d_ff, d_model))         # ff2
total_forward_flops = n_layers * per_layer_flops

print(f"\nPer-token forward FLOPs, explicit sum over every matmul: {total_forward_flops:,}")
print(f"2 * N_matmuls (the standard '2N' approximation):          {2 * N_matmuls:,}")
print(f"Exact match (not an approximation, for the matmul-dominated part): "
      f"{total_forward_flops == 2 * N_matmuls}")
print(f"Full training step (forward + backward), the '6N' rule Chinchilla's C=6ND (05.07) uses:")
print(f"  6 * N_matmuls = {6 * N_matmuls:,} FLOPs per token")

# --- Memory: training (Adam) vs inference (weights + KV cache) ---
def bytes_for_dtype(dtype):
    return {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1}[dtype]

def training_memory_bytes(N, dtype="fp32"):
    # Adam needs 4 same-sized buffers: params, gradients, momentum m, variance v
    return N * 4 * bytes_for_dtype(dtype)

def inference_weight_memory_bytes(N, dtype="fp16"):
    return N * bytes_for_dtype(dtype)

def kv_cache_bytes(n_layers, n_heads, d_head, seq_len, batch_size, dtype="fp16"):
    return 2 * n_layers * n_heads * d_head * seq_len * batch_size * bytes_for_dtype(dtype)

N_7b = 7e9
n_heads_7b, d_model_7b, n_layers_7b = 32, 4096, 32
d_head_7b = d_model_7b // n_heads_7b

train_mem = training_memory_bytes(N_7b, "fp32")
inf_mem = inference_weight_memory_bytes(N_7b, "fp16")
print(f"\nAt a 7B-parameter scale (a common 'small' modern LLM size):")
print(f"  Training memory (Adam, fp32): {train_mem/1e9:.1f} GB "
      f"(params={N_7b*4/1e9:.1f} + grad={N_7b*4/1e9:.1f} + m={N_7b*4/1e9:.1f} + v={N_7b*4/1e9:.1f} GB)")
print(f"  Inference weight memory (fp16): {inf_mem/1e9:.1f} GB")
print(f"  Training-to-inference-weights ratio: {train_mem/inf_mem:.0f}x")

for seq_len, batch_size in [(2048, 1), (2048, 8), (8192, 1)]:
    kv_mem = kv_cache_bytes(n_layers_7b, n_heads_7b, d_head_7b, seq_len, batch_size, "fp16")
    print(f"  KV cache (seq_len={seq_len}, batch={batch_size}): {kv_mem/1e9:.2f} GB")

## 🤖 Why This Matters for AI

Every number derived above turns directly into a hardware purchasing or deployment decision.
Applying the memory formulas to three real model scales (7B, 13B, 70B — common sizes in the
current open-model ecosystem) against real GPU memory specifications (RTX 4090: 24GB, A100:
40GB/80GB, H100: 80GB) shows exactly why a 70B model needs multiple GPUs working together
(its weights alone exceed any single card's memory) while a 7B model comfortably fits on
consumer hardware even with a substantial KV cache. This is also exactly the calculation behind
a common real deployment question: "how many concurrent requests can this GPU serve?" — since
the KV cache (not the fixed weight memory) is what grows with batch size and context length,
there's a hard ceiling on how many sequences a given GPU can serve simultaneously once the
weights are loaded, computed directly below for a concrete case.

In [ ]:
gpu_memory_gb = {"RTX 4090": 24, "A100 40GB": 40, "A100 80GB": 80, "H100": 80}

model_configs = {
    "7B":  dict(N=7e9,  n_layers=32, d_model=4096, n_heads=32),
    "13B": dict(N=13e9, n_layers=40, d_model=5120, n_heads=40),
    "70B": dict(N=70e9, n_layers=80, d_model=8192, n_heads=64),
}

print("Inference memory (weights + 4k-context KV cache, batch=1) vs. real GPU memory:")
results = {}
for model_name, cfg in model_configs.items():
    N = cfg["N"]
    weight_mem_gb = inference_weight_memory_bytes(N, "fp16") / 1e9
    d_head = cfg["d_model"] // cfg["n_heads"]
    kv_mem_gb = kv_cache_bytes(cfg["n_layers"], cfg["n_heads"], d_head, seq_len=4096, batch_size=1, dtype="fp16") / 1e9
    total_gb = weight_mem_gb + kv_mem_gb
    fits_on = [gpu for gpu, mem in gpu_memory_gb.items() if mem >= total_gb]
    results[model_name] = (weight_mem_gb, kv_mem_gb, total_gb, fits_on)
    print(f"  {model_name:>4}: weights={weight_mem_gb:6.1f}GB, KV cache={kv_mem_gb:5.2f}GB, "
          f"total={total_gb:6.1f}GB -- fits on: {fits_on if fits_on else '(none of these)'}")

# How many concurrent 4k-context requests can an 80GB GPU serve for the 7B model?
model_7b = model_configs["7B"]
weight_mem = inference_weight_memory_bytes(model_7b["N"], "fp16") / 1e9
available_for_kv = 80 - weight_mem
d_head_7b = model_7b["d_model"] // model_7b["n_heads"]
kv_per_seq = kv_cache_bytes(model_7b["n_layers"], model_7b["n_heads"], d_head_7b,
                             seq_len=4096, batch_size=1, dtype="fp16") / 1e9
max_batch = int(available_for_kv / kv_per_seq)
print(f"\n7B model on an 80GB GPU: {weight_mem:.1f}GB fixed for weights, "
      f"{available_for_kv:.1f}GB remaining for KV cache")
print(f"KV cache per sequence (4k context): {kv_per_seq:.3f}GB")
print(f"Maximum CONCURRENT 4k-context requests this GPU can serve: {max_batch}")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
names = list(model_configs.keys())
weights = [results[m][0] for m in names]
kv = [results[m][1] for m in names]
x = np.arange(len(names))
ax.bar(x, weights, label="weights (fp16)", color="#4C72B0")
ax.bar(x, kv, bottom=weights, label="KV cache (4k context, batch=1)", color="#C44E52")
for gpu, mem in gpu_memory_gb.items():
    ax.axhline(mem, ls=':', lw=1, color='gray')
    ax.text(len(names) - 0.4, mem + 1, gpu, fontsize=7, color='gray')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("memory (GB)")
ax.set_title("Model memory footprint vs. real GPU memory capacities\n(dotted lines mark actual GPU VRAM sizes)")
ax.legend()
plt.show()

## 🏋️ Exercises

### Warm-up
1. By hand, using the formula $12\,n_{\text{layers}}\,d_{\text{model}}^2$, compute the
   (non-embedding) parameter count for a model with $n_{\text{layers}}=24$,
   $d_{\text{model}}=1024$ — GPT-2-medium's configuration. Verify against the Implementation
   section's `N_matmuls` calculation with these values substituted in, and check your total
   (plus embedding) comes within 1% of the published 355M figure the Visual Intuition section
   uses.

### Practice
2. In the Implementation section, `linear_flops` assumes a dense (fully-connected) matmul.
   Attention also computes $QK^\top$ and (attention weights)$\times V$ — two matmuls whose cost
   scales with *sequence length*, not just $d_{\text{model}}$, and which the "$2N$" formula
   above quietly ignores (it only counts the $Q,K,V,O$ *projection* matmuls, not the attention
   score computation itself). Add the FLOPs for $QK^\top$ and $AV$ (each costing roughly
   $2\cdot\text{seq\_len}\cdot d_{\text{model}}$ per token, for one layer) to
   `total_forward_flops` for a sequence length of 2048, and report by what percentage the total
   changes. At what sequence length does this "ignored" cost become comparable to the
   projection matmuls' cost, for GPT-2-small's $d_{\text{model}}=768$?

### Challenge
3. Real inference serving often uses **quantization** (e.g. int8 or even int4 weights) to
   reduce memory further. Add an `"int4"` option to `bytes_for_dtype` (0.5 bytes per parameter)
   and recompute the 70B model's total inference memory (weights + 4k-context KV cache) at
   int4 weights. Does it now fit on a single 80GB GPU? Note that `kv_cache_bytes` above still
   uses fp16 for the KV cache regardless of weight quantization — research (or reason from the
   formulas) why KV cache quantization is a separate, distinct decision from weight
   quantization, and what accuracy risk it introduces that weight quantization alone does not.

---

## 📚 Further Reading
- Kaplan et al., [\"Scaling Laws for Neural Language Models\"](https://arxiv.org/abs/2001.08361)
- Narayanan et al., [\"Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM\"](https://arxiv.org/abs/2104.04473)
- Pope et al., [\"Efficiently Scaling Transformer Inference\"](https://arxiv.org/abs/2211.05102)

---

*Next notebook: [Capstone: Read a Research Paper Mathematically](05_read_a_research_paper.ipynb)*